# TODO 6.5：FP16 / INT8 同硬件 Benchmark

在同一台 A100 上使用相同 16 点矩阵运行 FP16 和 SmoothQuant INT8，消除跨硬件、跨测试矩阵造成的不可比问题。

In [1]:
%pip -q install -U "nvidia-modelopt[hf]" accelerate safetensors
print('安装完成后请重启 Colab 会话，再从下一单元继续。')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 69.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.7/51.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 6.7 MB/s eta 0:00:00
安装完成后请重启 Colab 会话，再从下一单元继续。


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, sys
from pathlib import Path
import torch

PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
INT8_MANIFEST = PROJECT_DIR / 'out/int8_fp8/smoothquant-int8-w8a8/quantized_model_manifest.json'
BASELINE_CONFIG = PROJECT_DIR / 'configs/baseline.json'
BENCHMARK_CONFIG = PROJECT_DIR / 'configs/quantization_benchmark.json'
COMPARISON_CONFIG = PROJECT_DIR / 'configs/quantization_comparison.json'
RESULT_DIR = PROJECT_DIR / 'out/quantization_comparison'
INT8_RESULT_DIR = PROJECT_DIR / 'out/int8_fp8'
sys.path.insert(0, str(PROJECT_DIR))

from src.baseline.service import BaselineService, InferenceRequest
from src.benchmark.runner import run_benchmark, validate_benchmark_report
from src.data.finqa_assets import load_assets
from src.quantization.comparison import build_quantization_comparison, render_markdown, validate_quantization_comparison
from src.quantization.int8_fp8 import load_smoothquant_service

assert torch.cuda.is_available(), '请切换到 NVIDIA A100 GPU。'
assert 'A100' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
assets = load_assets(DATASET_DIR)
benchmark_config = json.loads(BENCHMARK_CONFIG.read_text(encoding='utf-8'))
prompt = InferenceRequest.from_finqa_row(assets.quality_dev[0], request_id='comparison-benchmark').render_prompt()
print('GPU:', torch.cuda.get_device_name(0))
print('测试点:', len(benchmark_config['matrix']['batch_sizes']) * len(benchmark_config['matrix']['input_lengths']) * len(benchmark_config['matrix']['output_lengths']))

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-80GB
测试点: 16


In [2]:
fp16_service = BaselineService.from_local_model(BASELINE_CONFIG, MODEL_MANIFEST, RESULT_DIR / 'fp16_a100_service.jsonl')
fp16_report = run_benchmark(fp16_service, prompt, benchmark_config)
fp16_path = RESULT_DIR / 'fp16_a100_benchmark_report.json'
fp16_path.write_text(json.dumps(fp16_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
fp16_validation = validate_benchmark_report(fp16_path)
(RESULT_DIR / 'fp16_a100_benchmark_validation.json').write_text(json.dumps(fp16_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert fp16_validation.complete
del fp16_service; gc.collect(); torch.cuda.empty_cache()
print('FP16 A100 Benchmark: PASS')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Benchmark 1/16: bs1-in128-out16
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
FP16 A100 Benchmark: PASS


In [3]:
int8_service = load_smoothquant_service(BASELINE_CONFIG, INT8_MANIFEST, MODEL_MANIFEST, INT8_RESULT_DIR / 'int8_benchmark_service.jsonl')
int8_report = run_benchmark(int8_service, prompt, benchmark_config)
int8_path = INT8_RESULT_DIR / 'benchmark_report.json'
int8_path.write_text(json.dumps(int8_report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
int8_validation = validate_benchmark_report(int8_path)
(INT8_RESULT_DIR / 'benchmark_validation.json').write_text(json.dumps(int8_validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
assert int8_validation.complete
del int8_service; gc.collect(); torch.cuda.empty_cache()
print('SmoothQuant INT8 A100 Benchmark: PASS')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Registered <class 'transformers.models.qwen2.modeling_qwen2.Qwen2Attention'> to _QuantAttention for KV Cache quantization
Inserted 606 quantizers
Benchmark 1/16: bs1-in128-out16
Loading extension modelopt_cuda_ext...
Loaded extension modelopt_cuda_ext in 59.5 seconds
Benchmark 2/16: bs1-in128-out32
Benchmark 3/16: bs1-in512-out16
Benchmark 4/16: bs1-in512-out32
Benchmark 5/16: bs4-in128-out16
Benchmark 6/16: bs4-in128-out32
Benchmark 7/16: bs4-in512-out16
Benchmark 8/16: bs4-in512-out32
Benchmark 9/16: bs8-in128-out16
Benchmark 10/16: bs8-in128-out32
Benchmark 11/16: bs8-in512-out16
Benchmark 12/16: bs8-in512-out32
Benchmark 13/16: bs16-in128-out16
Benchmark 14/16: bs16-in128-out32
Benchmark 15/16: bs16-in512-out16
Benchmark 16/16: bs16-in512-out32
SmoothQuant INT8 A100 Benchmark: PASS


In [4]:
report = build_quantization_comparison(PROJECT_DIR, COMPARISON_CONFIG)
report_path = RESULT_DIR / 'comparison_report.json'
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
(RESULT_DIR / 'comparison_report.md').write_text(render_markdown(report), encoding='utf-8')
validation = validate_quantization_comparison(report_path)
(RESULT_DIR / 'validation.json').write_text(json.dumps(validation.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(validation.to_dict(), ensure_ascii=False, indent=2))
print('结果目录:', RESULT_DIR)

{
  "report": "/content/drive/MyDrive/OptimizationSystem/out/quantization_comparison/comparison_report.json",
  "complete": false,
  "checks": [
    {
      "name": "fp16_int8_int4_fp8_comparison_table",
      "status": "pass",
      "detail": "FP16, INT8, INT4, and FP8 rows are present."
    },
    {
      "name": "quality_changes_recorded",
      "status": "pass",
      "detail": "Every candidate has measured quality."
    },
    {
      "name": "memory_changes_recorded",
      "status": "not_run",
      "detail": "Every candidate has measured memory."
    },
    {
      "name": "latency_changes_recorded",
      "status": "not_run",
      "detail": "Every candidate has measured latency."
    },
    {
      "name": "throughput_changes_recorded",
      "status": "not_run",
      "detail": "Every candidate has measured throughput."
    },
    {
      "name": "unit_token_cost_recorded",
      "status": "not_run",
      "detail": "Every measured candidate has an approved monetary cost."
 